# Job A — Feature-based Real-IAD benchmark (Colab)

Runs PatchCore + PaDiM + SubspaceAD on 30 Real-IAD categories, camera C1 only.

**Before running:**
- Runtime -> Change runtime type -> GPU (T4/V100/A100).
- `Real-IAD_dataset/realiad_1024/*.zip` must be on Drive (zipped, one per category).

**Outputs:** synced to `/content/drive/MyDrive/thesis_runs/jobA/` per-category. Safe to
interrupt and restart — finished categories get a `<category>.done` marker and are skipped on resume.

In [ ]:
# 1. Mount Drive and verify the Real-IAD zips are visible.
from google.colab import drive
drive.mount('/content/drive')

import os, glob
ZIPS_DIR = '/content/drive/MyDrive/Real-IAD_dataset/realiad_1024'
zips = sorted(glob.glob(os.path.join(ZIPS_DIR, '*.zip')))
print(f'Found {len(zips)} category zips under {ZIPS_DIR}')
for z in zips[:5]:
    print('  -', os.path.basename(z))
assert len(zips) > 0, 'No zips found — check ZIPS_DIR matches your Drive layout.'

In [ ]:
# 2. Clone (or pull) the thesis repo into /content/.
# Replace <your-username>/<your-repo> with the repo you pushed to.
REPO_URL = 'https://github.com/LorenzSF/Real-time-visual-defect-detection.git'
REPO_DIR = '/content/Real-time-visual-defect-detection'
BRANCH   = 'main'

import os, subprocess
if os.path.isdir(REPO_DIR):
    subprocess.check_call(['git', '-C', REPO_DIR, 'fetch', '--all'])
    subprocess.check_call(['git', '-C', REPO_DIR, 'checkout', BRANCH])
    subprocess.check_call(['git', '-C', REPO_DIR, 'pull', '--ff-only'])
else:
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR])

print('Repo ready at', REPO_DIR)
print(subprocess.check_output(['git', '-C', REPO_DIR, 'log', '-1', '--oneline']).decode().strip())

In [ ]:
# 3. Install Python dependencies for the three models.
# Anomalib adapters require anomalib >= 2.x (module path `anomalib.models.image.*`).
# Anomalib 2.x dropped `imgaug` which breaks on NumPy 2 (Colab default).
# transformers: pinned to the 4.x branch >= 4.46 for DINOv2-with-registers support;
# upper bound <5 avoids breaking anomalib 2.3.x which was not tested against 5.x.
!pip install -q "anomalib>=2.2,<3" lightning "transformers>=4.46,<5" scikit-learn timm kornia einops
!pip install -q -e /content/Real-time-visual-defect-detection --no-deps || echo '[note] editable install skipped (not required)'

import torch, anomalib, lightning, transformers
print('torch       ', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')
print('anomalib    ', anomalib.__version__)
print('lightning   ', lightning.__version__)
print('transformers', transformers.__version__)

In [ ]:
# 4. Launch the category loop. Streams logs live; safe to interrupt and rerun.
import os
os.environ['ZIPS_DIR']    = '/content/drive/MyDrive/Real-IAD_dataset/realiad_1024'
os.environ['RESULTS_DIR'] = '/content/drive/MyDrive/thesis_runs/jobA'
os.environ['WORK_DIR']    = '/content/work'
os.environ['REPO_DIR']    = '/content/Real-time-visual-defect-detection'
os.environ['CONFIG']      = '/content/Real-time-visual-defect-detection/src/benchmark_AD/configs/colab_featurebased.yaml'

# Ensure the Drive log dir exists before `tee` tries to append to it.
!mkdir -p {os.environ['RESULTS_DIR']}
!bash /content/Real-time-visual-defect-detection/scripts/run_jobA_colab.sh 2>&1 | tee -a {os.environ['RESULTS_DIR']}/run.log

## Troubleshooting

- **`No Real-IAD images found`**: the zip did not unpack with the expected `OK/` + `NG/<defect>/S000X/` layout. Inspect one extracted category in `/content/work/<category>/` and share the `ls` output.
- **`CUDA out of memory` on SubspaceAD**: lower `subspacead.batch_size` in the config (default 4) to 2.
- **Session disconnected mid-run**: reopen the notebook, re-run cells 1-3, then rerun cell 4 — completed categories are skipped via `.done` markers.
- **Drive I/O stalls**: check `Runtime -> Manage sessions`; if Drive is throttled, wait a few minutes or reconnect.

In [ ]:
# 6. Release Colab resources once the run is finished and synced.
# This clears local caches first, then disconnects the runtime.
import gc

try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception:
    pass

gc.collect()

try:
    from google.colab import runtime
    runtime.unassign()
except Exception:
    pass
